# NumPy Fundamentals for AI

> **Interview Note:** NumPy is the foundation of PyTorch, TensorFlow, JAX, and all scientific Python. Understanding arrays, broadcasting, memory layout, and vectorization is non-negotiable for AI engineering.

---

## 1. Array Creation & Basics

In [ ]:
import numpy as np

# 1D array (vector)
v = np.array([1, 2, 3, 4])
print(f"Vector: {v}, shape: {v.shape}, ndim: {v.ndim}, dtype: {v.dtype}")

# 2D array (matrix)
m = np.array([[1, 2], [3, 4], [5, 6]])
print(f"Matrix:\n{m}\nshape: {m.shape}")

# 3D array (tensor)
t = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print(f"Tensor shape: {t.shape}")  # (2, 2, 2)

# Special arrays
print(f"Zeros: {np.zeros((3, 4))}")
print(f"Ones: {np.ones((2, 3))}")
print(f"Full: {np.full((2, 2), 7)}")
print(f"Eye: {np.eye(3)}")
print(f"Arange: {np.arange(0, 10, 2)}")
print(f"Linspace: {np.linspace(0, 1, 5)}")
print(f"Random: {np.random.randn(2, 3)}")
print(f"Random int: {np.random.randint(0, 10, (2, 3))}")

---

## 2. Data Types (dtypes) — Critical for Memory & Precision

In [ ]:
# Common dtypes in AI
dtypes = [
    np.int8, np.int16, np.int32, np.int64,
    np.uint8, np.uint16,
    np.float16, np.float32, np.float64,
    np.bool_, np.complex64
]

for dt in dtypes:
    arr = np.array([1], dtype=dt)
    print(f"{str(dt):12} | {arr.itemsize} bytes | range: {np.iinfo(dt).min if np.issubdtype(dt, np.integer) else 'N/A'} ... {np.iinfo(dt).max if np.issubdtype(dt, np.integer) else 'N/A'}")

# Float precision
print(f"\nfloat16 eps: {np.finfo(np.float16).eps:.2e}")
print(f"float32 eps: {np.finfo(np.float32).eps:.2e}")
print(f"float64 eps: {np.finfo(np.float64).eps:.2e}")

# Casting
x = np.array([1.5, 2.7, 3.9])
print(f"\nOriginal: {x} ({x.dtype})")
print(f"As int32: {x.astype(np.int32)}")
print(f"As float16: {x.astype(np.float16)}")

> **AI Context:** Training uses `float32` (or `bfloat16`/`float16` with mixed precision). Inference can use `int8` quantization. `float64` is rarely used in DL (2x memory, slower on GPU).

---

## 3. Memory Layout: C vs Fortran Order

In [ ]:
# Row-major (C-style) - DEFAULT
c_arr = np.array([[1, 2, 3], [4, 5, 6]], order='C')
print(f"C-order: {c_arr.flags['C_CONTIGUOUS']}, F-order: {c_arr.flags['F_CONTIGUOUS']}")
print(f"Strides: {c_arr.strides}")  # (24, 8) bytes - row stride, col stride

# Column-major (Fortran-style)
f_arr = np.array([[1, 2, 3], [4, 5, 6]], order='F')
print(f"\nF-order: {f_arr.flags['C_CONTIGUOUS']}, F-order: {f_arr.flags['F_CONTIGUOUS']}")
print(f"Strides: {f_arr.strides}")  # (8, 16) bytes

# Why it matters: slicing creates views, not copies
print(f"\nRow slice (view): {c_arr[0].base is c_arr}")
print(f"Col slice (copy): {c_arr[:, 0].base is c_arr}")

# Transpose swaps strides (no copy!)
t = c_arr.T
print(f"\nTranspose strides: {t.strides}")
print(f"Transpose is view: {t.base is c_arr}")

> **Interview Question:** "What's the difference between `arr.T` and `arr.transpose()`?" → `T` is property (2D only), `transpose()` accepts axes. Both return views when possible.

---

## 4. Indexing & Slicing

In [ ]:
arr = np.arange(24).reshape(2, 3, 4)  # (batch, seq, features)
print(f"Shape: {arr.shape}")
print(arr)

# Basic indexing
print(f"\narr[0]: {arr[0].shape}")        # (3, 4) - first batch
print(f"arr[0, 1]: {arr[0, 1].shape}")    # (4,) - first batch, second seq
print(f"arr[0, 1, 2]: {arr[0, 1, 2]}")    # scalar

# Slicing
print(f"\narr[:, 1:3]: {arr[:, 1:3].shape}")  # all batches, seq 1-2
print(f"arr[..., 1]: {arr[..., 1].shape}")    # all but last dim, feature 1
print(f"arr[0, ..., :2]: {arr[0, ..., :2].shape}")

# Fancy indexing (returns COPY, not view!)
indices = [0, 2]
print(f"\narr[[0, 2]]: {arr[[0, 2]].shape}")  # batches 0 and 2
print(f"arr[:, [0, 2]]: {arr[:, [0, 2]].shape}")

# Boolean indexing
mask = arr[0, :, 0] > 4
print(f"\nMask: {mask}")
print(f"arr[0][mask]: {arr[0][mask]}")

---

## 5. Broadcasting — The Superpower

In [ ]:
# Broadcasting rules:
# 1. Align dimensions from RIGHT
# 2. Dimensions must be equal OR one is 1
# 3. Missing dimensions treated as 1

# Examples
a = np.array([[1, 2, 3], [4, 5, 6]])  # (2, 3)
b = np.array([10, 20, 30])             # (3,)

print(f"a + b:\n{a + b}")  # b broadcast to (2, 3)

# Add column vector
c = np.array([[10], [20]])  # (2, 1)
print(f"a + c:\n{a + c}")  # c broadcast to (2, 3)

# Practical: normalize by mean/std per feature
data = np.random.randn(100, 10)
mean = data.mean(axis=0)  # (10,)
std = data.std(axis=0)    # (10,)
normalized = (data - mean) / std  # Broadcasting!
print(f"\nNormalized mean: {normalized.mean(axis=0)[:3]}")
print(f"Normalized std: {normalized.std(axis=0)[:3]}")

# Outer product via broadcasting
x = np.array([1, 2, 3])
y = np.array([10, 20])
print(f"\nOuter product:\n{x[:, None] * y[None, :]}")

> **Interview Question:** "When does broadcasting fail?" → When aligned dimensions are incompatible (neither equal nor 1).

---

## 6. Vectorization vs Loops

In [ ]:
import time

size = 10_000_000
a = np.random.randn(size)
b = np.random.randn(size)

# Loop (slow)
start = time.perf_counter()
c_loop = np.empty(size)
for i in range(size):
    c_loop[i] = a[i] * b[i] + a[i]
loop_time = time.perf_counter() - start

# Vectorized (fast)
start = time.perf_counter()
c_vec = a * b + a
vec_time = time.perf_counter() - start

print(f"Loop: {loop_time:.3f}s")
print(f"Vectorized: {vec_time:.3f}s")
print(f"Speedup: {loop_time/vec_time:.0f}x")
print(f"Results match: {np.allclose(c_loop, c_vec)}")

> **Key Insight:** Vectorization pushes loops to C/Fortran/BLAS. NumPy uses SIMD (AVX/SSE) and multi-threading (via BLAS: OpenBLAS, MKL, BLIS).

---

## 7. Essential Operations for AI

In [ ]:
# Matrix multiplication
A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C = A @ B  # or np.matmul(A, B)
print(f"Matmul: {A.shape} @ {B.shape} = {C.shape}")

# Batched matmul
batch_A = np.random.randn(10, 3, 4)
batch_B = np.random.randn(10, 4, 5)
batch_C = batch_A @ batch_B  # (10, 3, 5)
print(f"Batched: {batch_A.shape} @ {batch_B.shape} = {batch_C.shape}")

# Einsum - explicit tensor contractions
# 'bij,bjk->bik' = batch matrix multiply
einsum_C = np.einsum('bij,bjk->bik', batch_A, batch_B)
print(f"Einsum matches: {np.allclose(batch_C, einsum_C)}")

# Attention-style: (batch, heads, seq, head_dim) @ (batch, heads, head_dim, seq)
Q = np.random.randn(2, 8, 16, 64)
K = np.random.randn(2, 8, 16, 64)
attn_scores = np.einsum('bhqd,bhkd->bhqk', Q, K)  # (2, 8, 16, 16)
print(f"Attention scores shape: {attn_scores.shape}")

# Reduction operations
x = np.random.randn(4, 5, 6)
print(f"\nSum all: {x.sum()}")
print(f"Sum axis=1: {x.sum(axis=1).shape}")
print(f"Mean axis=(0,2): {x.mean(axis=(0,2)).shape}")
print(f"Max keepdims: {x.max(axis=1, keepdims=True).shape}")

---

## 8. Reshaping & Dimension Manipulation

In [ ]:
x = np.arange(24).reshape(2, 3, 4)
print(f"Original: {x.shape}")

# Reshape (view when possible)
print(f"Flatten: {x.reshape(-1).shape}")
print(f"Reshape: {x.reshape(3, 8).shape}")
print(f"Reshape: {x.reshape(2, -1).shape}")

# Transpose / permute axes
print(f"\nTranspose: {x.transpose(2, 0, 1).shape}")
print(f"Move axis: {np.moveaxis(x, 0, -1).shape}")
print(f"Swap axes: {np.swapaxes(x, 1, 2).shape}")

# Add/remove dimensions
print(f"\nExpand dims: {np.expand_dims(x, 0).shape}")
print(f"Squeeze: {np.squeeze(np.expand_dims(x, 0)).shape}")

# Flatten variants
print(f"\nRavel (view): {x.ravel().shape}, base is x: {x.ravel().base is x}")
print(f"Flatten (copy): {x.flatten().shape}, base is x: {x.flatten().base is x}")

---

## 9. Random Sampling (Reproducibility)

In [ ]:
# Modern random API (NumPy 1.17+)
rng = np.random.default_rng(42)  # Seeded generator

print(f"Normal: {rng.normal(0, 1, (2, 3))}")
print(f"Uniform: {rng.uniform(0, 1, (2, 3))}")
print(f"Integers: {rng.integers(0, 10, (2, 3))}")
print(f"Choice: {rng.choice([1,2,3,4,5], size=3, replace=False)}")
print(f"Permutation: {rng.permutation(5)}")

# Shuffle in-place
arr = np.arange(10)
rng.shuffle(arr)
print(f"Shuffled: {arr}")

# Multiple independent streams
rng1 = np.random.default_rng(1)
rng2 = np.random.default_rng(2)
print(f"\nStream 1: {rng1.random(3)}")
print(f"Stream 2: {rng2.random(3)}")

> **Best Practice:** Use `default_rng(seed)` instead of global `np.random.seed()`. Thread-safe, independent streams, better algorithms (PCG64).

---

## 10. File I/O

In [ ]:
# .npy - single array
arr = np.random.randn(100, 100)
np.save('data.npy', arr)
loaded = np.load('data.npy')
print(f".npy: {loaded.shape}, match: {np.array_equal(arr, loaded)}")

# .npz - multiple arrays (compressed)
np.savez_compressed('data.npz', weights=arr, bias=np.ones(100))
loaded = np.load('data.npz')
print(f".npz keys: {list(loaded.keys())}")
print(f"weights shape: {loaded['weights'].shape}")

# Memory-mapped arrays (for large files > RAM)
mmap = np.lib.format.open_memmap('data.npy', mode='r')
print(f"\nMemmap: {mmap.shape}, dtype: {mmap.dtype}")
print(f"First row: {mmap[0, :5]}")

---

## 11. Common Pitfalls & Gotchas

In [ ]:
# 1. Copy vs View
a = np.array([1, 2, 3])
b = a           # Same object
c = a.view()    # View (shared data)
d = a.copy()    # Deep copy

b[0] = 99
print(f"After b[0]=99: a={a}, b={b}, c={c}, d={d}")

# 2. Integer division
print(f"\n5 / 2 = {5 / 2}")        # float
print(f"5 // 2 = {5 // 2}")       # floor
print(f"np.array([5]) / 2 = {np.array([5]) / 2}")
print(f"np.array([5]) // 2 = {np.array([5]) // 2}")

# 3. Boolean indexing with assignment
x = np.array([1, 2, 3, 4, 5])
x[x > 3] = 0
print(f"\nAfter x[x>3]=0: {x}")

# 4. NaN comparisons
nan_arr = np.array([1.0, np.nan, 3.0])
print(f"\nNaN == NaN: {np.nan == np.nan}")
print(f"np.isnan: {np.isnan(nan_arr)}")
print(f"Any NaN: {np.any(np.isnan(nan_arr))}")

# 5. Axis confusion
m = np.array([[1, 2, 3], [4, 5, 6]])  # (2, 3)
print(f"\nAxis 0 (rows): {m.sum(axis=0)}")   # Sum DOWN rows → (3,)
print(f"Axis 1 (cols): {m.sum(axis=1)}")   # Sum ACROSS cols → (2,)

---

## 12. Performance Tips

In [ ]:
# 1. Pre-allocate, don't append
# BAD:
result = np.array([])
for i in range(1000):
    result = np.append(result, i)  # O(n²) - copies every time!

# GOOD:
result = np.empty(1000)
for i in range(1000):
    result[i] = i

# BETTER: vectorize entirely
result = np.arange(1000)

# 2. Use in-place operations
a = np.random.randn(1000, 1000)
b = np.random.randn(1000, 1000)

import time
start = time.perf_counter()
c = a + b  # New allocation
print(f"New allocation: {time.perf_counter() - start:.4f}s")

start = time.perf_counter()
np.add(a, b, out=a)  # In-place
print(f"In-place: {time.perf_counter() - start:.4f}s")

# 3. Avoid Python loops in hot paths — use:
# - np.where, np.searchsorted, np.digitize
# - np.apply_along_axis (still has Python overhead)
# - Numba (@njit) for custom loops
# - Or write in PyTorch/JAX which fuse kernels

---

## 13. NumPy → PyTorch / TensorFlow / JAX

In [ ]:
# NumPy → PyTorch
try:
    import torch
    np_arr = np.random.randn(3, 4)
    torch_tensor = torch.from_numpy(np_arr)  # Shares memory!
    print(f"torch.from_numpy: {torch_tensor.shape}, dtype: {torch_tensor.dtype}")
    
    # Modifying numpy affects tensor and vice versa
    np_arr[0, 0] = 999
    print(f"After numpy change: {torch_tensor[0, 0]}")
    
    # Tensor → NumPy
    tensor = torch.randn(3, 4)
    back_to_np = tensor.numpy()  # Shares memory if CPU
    print(f"tensor.numpy(): {back_to_np.shape}")
    
    # For GPU tensors: .cpu().numpy()
    # gpu_tensor.cpu().numpy()
    
except ImportError:
    print("PyTorch not installed")

# JAX
try:
    import jax.numpy as jnp
    np_arr = np.random.randn(3, 4)
    jax_arr = jnp.array(np_arr)  # Copy
    print(f"JAX array: {jax_arr.shape}")
    back = np.array(jax_arr)  # Copy back
except ImportError:
    print("JAX not installed")

---

## 14. Interview Questions

1. **What's the difference between a view and a copy?**
   - View shares memory (strides change), copy allocates new memory
   - Slicing → view, fancy indexing → copy, `.copy()` → copy

2. **Explain broadcasting rules.**
   - Align right, dims must match or be 1, missing dims = 1

3. **What are strides? How does transpose affect them?**
   - Strides = bytes to step in each dimension. Transpose swaps strides (no data copy).

4. **Why is `float32` standard in deep learning?**
   - 2x memory vs `float64`, faster on GPU, sufficient precision for gradients

5. **What's the difference between `np.dot`, `@`, `np.matmul`, `np.einsum`?**
   - `@` / `matmul`: Matrix multiply with broadcasting (preferred)
   - `dot`: Dot product, different broadcasting rules
   - `einsum`: Explicit index notation, most flexible

6. **How to normalize a batch of images (N, H, W, C) by per-channel mean/std?**
   - `mean = img.mean(axis=(0,1,2), keepdims=True)` → shape (1,1,1,C)
   - Broadcasts correctly: `(img - mean) / std`

7. **What does `np.newaxis` / `None` do?**
   - Adds dimension: `arr[None, :]` → (1, N), `arr[:, None]` → (N, 1)

8. **How to efficiently compute pairwise distances?**
   - `np.sum((x[:, None] - y[None, :])**2, axis=-1)` or `scipy.spatial.distance.cdist`

9. **What's the memory cost of `np.concatenate([arr]*1000)`?**
   - Creates 1000 copies! Use `np.tile(arr, (1000,))` or `np.broadcast_to` for views.

10. **How to set random seed for reproducibility across runs?**
    - `rng = np.random.default_rng(seed)` — use this generator instance everywhere
    - Avoid global `np.random.seed()` (not thread-safe, affects all code)